# Create a `VOCAB` table from the Brandon Sanderson `CORPUS`
Columns as delimitted names, including `n`, `p`', `i`, `dfidf`, `porter_stem`, `max_pos` and `max_pos_group`, `stop`

## Get Data

In [7]:
import pandas as pd
import numpy as np
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords

nltk.download('stopwords')

directory_path  = 'C:/Users/mason/Box/Text As Data Final/Output' 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mason\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [8]:
CORPUS = pd.read_csv(directory_path + '/BrandonSanderson_CORPUS.csv')
CORPUS.head()

,Unnamed: 0,title,chapter_id,paragraph_id,sent_id,token_id,token_str,term_str,pos,pos_group
0,0,A Memory of Light,0,0,0,0,The,the,DT,DT
1,1,A Memory of Light,0,0,0,1,Wheel,wheel,NNP,NN
2,2,A Memory of Light,0,0,0,2,of,of,IN,IN
3,3,A Memory of Light,0,0,0,3,Time,time,NNP,NN
4,4,A Memory of Light,0,0,0,4,turns,turns,NNS,NN


## Lets get 'n', 'p', and 'i' first

In [9]:
CORPUS['term_str'] = CORPUS['term_str'].astype(str)
CORPUS = CORPUS[CORPUS['term_str'] != '']

VOCAB = CORPUS['term_str'].value_counts().to_frame('n')
VOCAB.index.name = 'term_str'
VOCAB['p'] = VOCAB['n'] / VOCAB['n'].sum()
VOCAB['i'] = -np.log2(VOCAB['p'])

VOCAB.head()



,n,p,i
term_str,,,
the,122074,0.054869,4.187866
to,59543,0.026763,5.223618
a,48864,0.021963,5.508778
of,44828,0.020149,5.633150
and,40305,0.018116,5.786591


## Get Document Frequencies

In [10]:
N_docs = CORPUS['title'].nunique()
document_freq = CORPUS.groupby('term_str')['title'].nunique()

VOCAB['df'] = document_freq
VOCAB['idf'] = np.log2(N_docs / VOCAB['df'])
VOCAB['dfidf'] = VOCAB['df'] * VOCAB['idf']

VOCAB.head()

,n,p,i,df,idf,dfidf
term_str,,,,,,
the,122074,0.054869,4.187866,24,0.0,0.0
to,59543,0.026763,5.223618,24,0.0,0.0
a,48864,0.021963,5.508778,24,0.0,0.0
of,44828,0.020149,5.633150,24,0.0,0.0
and,40305,0.018116,5.786591,24,0.0,0.0


## Add Max_POS and Max_pos_group

In [12]:
VOCAB['max_pos'] = CORPUS.groupby(['term_str', 'pos']).size().unstack(fill_value=0).idxmax(axis=1)
VOCAB['max_pos_group'] = CORPUS.groupby(['term_str', 'pos_group']).size().unstack(fill_value=0).idxmax(axis=1)
VOCAB.head()

,n,p,i,df,idf,dfidf,max_pos,max_pos_group
term_str,,,,,,,,
the,122074,0.054869,4.187866,24,0.0,0.0,DT,DT
to,59543,0.026763,5.223618,24,0.0,0.0,TO,TO
a,48864,0.021963,5.508778,24,0.0,0.0,DT,DT
of,44828,0.020149,5.633150,24,0.0,0.0,IN,IN
and,40305,0.018116,5.786591,24,0.0,0.0,CC,CC


## Stemming and Stop Words

In [13]:
stemmer = PorterStemmer()
VOCAB['porter_stem'] = VOCAB.index.to_series().apply(stemmer.stem)

sw = set(stopwords.words('english'))

VOCAB['stop'] = VOCAB.index.isin(sw).astype(int)

VOCAB = VOCAB[['n', 'p', 'i', 'dfidf', 'porter_stem', 'max_pos', 'max_pos_group', 'stop']]

In [14]:
VOCAB.to_csv(f"{directory_path}/BrandonSanderson_VOCAB.csv")
with pd.option_context('display.max_rows', 20):
    print(VOCAB.head(20))

               n         p         i     dfidf porter_stem max_pos  \
term_str                                                             
the       122074  0.054869  4.187866  0.000000         the      DT   
to         59543  0.026763  5.223618  0.000000          to      TO   
a          48864  0.021963  5.508778  0.000000           a      DT   
of         44828  0.020149  5.633150  0.000000          of      IN   
and        40305  0.018116  5.786591  0.000000         and      CC   
he         35776  0.016080  5.958558  1.412213          he     PRP   
it         27743  0.012470  6.325426  0.000000          it     PRP   
i          27738  0.012467  6.325686  0.000000           i     PRP   
you        26286  0.011815  6.403255  0.000000         you     PRP   
in         26140  0.011749  6.411291  0.000000          in      IN   
was        24937  0.011209  6.479262  0.000000          wa     VBD   
that       24734  0.011117  6.491054  0.000000        that      IN   
his        22747  0.